# Cross-species embedding — TranscriptFormer Metazoa

Objective: embed the cat/tiger benchmark after both species were remapped to
mouse genes, then compute the same scIB scores as the original notebook.

- Model: `transcriptformer==0.6.1`, checkpoint `tf-metazoa`.
- Input: raw, unnormalised counts with mouse Ensembl gene IDs.
- Benchmark: `orig.ident` as batch and `cell_type_ontology_term_id` as label.
- Baselines: PCA and a deterministic random embedding.

## Reproducible environment and model download

Run once on a Jean Zay login node:

```bash
export WORK="${WORK:-/lustre/fswork/projects/rech/xeg/$USER}"
export TF_ENV="$WORK/venvs/transcriptformer-0.6.1"
export UV="$HOME/.local/bin/uv"
export PYTHON_311="/gpfslocalsup/pub/anaconda-py3/2023.09/envs/python-3.11.5/bin/python"
export XDG_CACHE_HOME="$WORK/.cache"
export MPLCONFIGDIR="$WORK/.cache/matplotlib"
export JUPYTER_DATA_DIR="$WORK/.jupyter"
export TMPDIR="$WORK/.tmp"
mkdir -p "$XDG_CACHE_HOME" "$MPLCONFIGDIR" "$JUPYTER_DATA_DIR" "$TMPDIR"

"$UV" venv --python "$PYTHON_311" "$TF_ENV"
"$UV" pip install --python "$TF_ENV/bin/python"           "transcriptformer==0.6.1" "torch==2.5.1"           "papermill>=2.6,<3" "ipykernel>=6,<7"           scib-metrics scanpy leidenalg
"$TF_ENV/bin/python" -m ipykernel install --user           --name transcriptformer           --display-name "Python (TranscriptFormer 0.6.1)"

mkdir -p "$WORK/models/transcriptformer"
# Manual-only checkpoint command; it is intentionally disabled here.
# source /etc/profile.d/proxy.sh
# "$TF_ENV/bin/transcriptformer" download tf-metazoa           --checkpoint-dir "$WORK/models/transcriptformer"
```

`tf-metazoa` includes mouse and extends `tf-exemplar` to 12 species. The
benchmark remains on the existing mouse-remapped input so it is directly
comparable with the earlier exemplar and scPRINT runs.

In [1]:
import os
from pathlib import Path

os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "WANDB_MODE": "offline",
    "WANDB_DISABLED": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "HTTP_PROXY": "http://127.0.0.1:9",
    "HTTPS_PROXY": "http://127.0.0.1:9",
    "ALL_PROXY": "http://127.0.0.1:9",
    "NO_PROXY": "",
})

import anndata as ad
import numpy as np
import scanpy as sc
from IPython.display import display
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

SEED = 42
rng = np.random.default_rng(SEED)

DATA_PATH = Path(
    "notebooks/scPRINT-2-repro-notebooks/data/task_3_embed.h5ad"
)
RESULT_ROOT = Path("data/results/cross_species_embedding")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_KEY = "orig.ident"
LABEL_KEY = "cell_type_ontology_term_id"
MOUSE_ONTOLOGY_ID = "NCBITaxon:10090"

if not DATA_PATH.exists():
    raise FileNotFoundError(DATA_PATH)

import importlib.metadata as metadata
import scipy.sparse as sp
import subprocess

WORK = Path(os.environ.get("WORK", f"/lustre/fswork/projects/rech/xeg/{os.environ['USER']}"))
CHECKPOINT_PATH = WORK / "models/transcriptformer/tf_metazoa"
PREPARED_PATH = RESULT_ROOT / "task_3_mouse_transcriptformer_metazoa_input.h5ad"
OUTPUT_PATH = RESULT_ROOT / "transcriptformer_metazoa_embeddings.h5ad"
SCORE_PATH = RESULT_ROOT / "transcriptformer_metazoa_scib.csv"

print("TranscriptFormer", metadata.version("transcriptformer"))
print("Checkpoint", CHECKPOINT_PATH)
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


/lustre/fswork/projects/rech/xeg/uat95fg/venvs/transcriptformer-0.6.1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TranscriptFormer 0.6.1
Checkpoint /lustre/fswork/projects/rech/xeg/uat95fg/models/transcriptformer/tf_metazoa


## Prepare raw mouse-count input

The shared benchmark file already contains integer counts and mouse Ensembl IDs
under `var["ensembl_gene_id"]`. This cell writes a cached TranscriptFormer input
with the required `var["ensembl_id"]` name and no ambiguous `.raw` matrix.

In [2]:
def prepare_transcriptformer_input(source_path: Path, target_path: Path) -> Path:
    """Validate and write a fully local TranscriptFormer inference input."""
    input_path = target_path if target_path.exists() else source_path
    adata = sc.read_h5ad(input_path)
    if "ensembl_id" not in adata.var:
        if "ensembl_gene_id" not in adata.var:
            raise KeyError("Expected var['ensembl_gene_id'] in the remapped benchmark")
        adata.var["ensembl_id"] = (
            adata.var["ensembl_gene_id"].astype(str).to_numpy()
        )
    adata.obs["organism_ontology_term_id"] = MOUSE_ONTOLOGY_ID
    adata.obs["assay"] = "unknown"

    sample = adata.X[: min(128, adata.n_obs)]
    values = sample.data if sp.issparse(sample) else np.asarray(sample).ravel()
    if values.size and (
        np.nanmin(values) < 0
        or not np.allclose(values, np.rint(values), rtol=0, atol=1e-6)
    ):
        raise ValueError("TranscriptFormer requires raw non-negative integer counts")

    adata.raw = None
    adata.write_h5ad(target_path, compression="lzf")
    return target_path


prepare_transcriptformer_input(DATA_PATH, PREPARED_PATH)

PosixPath('data/results/cross_species_embedding/task_3_mouse_transcriptformer_metazoa_input.h5ad')

## GPU inference

Cell embeddings are read from the official `obsm["embeddings"]` output. The
output is cached so a restarted job skips completed inference.

In [3]:
if not OUTPUT_PATH.exists():
    command = [
        "transcriptformer",
        "inference",
        "--checkpoint-path",
        str(CHECKPOINT_PATH),
        "--data-file",
        str(PREPARED_PATH),
        "--output-path",
        str(RESULT_ROOT),
        "--output-filename",
        OUTPUT_PATH.name,
        "--gene-col-name",
        "ensembl_id",
        "--use-raw",
        "False",
        "--emb-type",
        "cell",
        "--device",
        "cuda",
        "--num-gpus",
        "1",
        "--precision",
        "16-mixed",
        "--batch-size",
        "1",
        "--oom-dataloader",
        "--n-data-workers",
        "4",
    ]
    print("Running:", " ".join(command), flush=True)
    subprocess.run(command, check=True)

output_da = sc.read_h5ad(OUTPUT_PATH)
prepared = sc.read_h5ad(PREPARED_PATH, backed="r")
if output_da.n_obs != prepared.n_obs:
    raise RuntimeError("TranscriptFormer changed the number of cells")
if not output_da.obs_names.equals(prepared.obs_names):
    # The TranscriptFormer CLI resets obs_names to 0..n-1. Verify that
    # stable metadata still identifies the same row order before restoring UUIDs.
    identity_columns = ["orig.ident", "batch", "cell_type", "cell_type_ontology_term_id"]
    for column in identity_columns:
        if column not in output_da.obs or column not in prepared.obs:
            raise KeyError(f"Missing identity column after TranscriptFormer: {column}")
        output_values = output_da.obs[column].reset_index(drop=True).astype(str)
        prepared_values = prepared.obs[column].reset_index(drop=True).astype(str)
        if not output_values.equals(prepared_values):
            raise RuntimeError(f"TranscriptFormer changed cell order ({column})")
    output_da.obs_names = prepared.obs_names.copy()
if "embeddings" not in output_da.obsm:
    raise KeyError("TranscriptFormer output has no obsm['embeddings']")
output_da.obsm["model_emb"] = np.asarray(output_da.obsm["embeddings"], dtype=np.float32)
output_da

Running: transcriptformer inference --checkpoint-path /lustre/fswork/projects/rech/xeg/uat95fg/models/transcriptformer/tf_metazoa --data-file data/results/cross_species_embedding/task_3_mouse_transcriptformer_metazoa_input.h5ad --output-path data/results/cross_species_embedding --output-filename transcriptformer_metazoa_embeddings.h5ad --gene-col-name ensembl_id --use-raw False --emb-type cell --device cuda --num-gpus 1 --precision 16-mixed --batch-size 1 --oom-dataloader --n-data-workers 4


2026-08-06 12:10:12,564 - INFO - Loading vocabulary file: /lustre/fswork/projects/rech/xeg/uat95fg/models/transcriptformer/tf_metazoa/vocabs/assay
2026-08-06 12:10:12,566 - INFO - Loading ESM2 mappings from /lustre/fswork/projects/rech/xeg/uat95fg/models/transcriptformer/tf_metazoa/vocabs


2026-08-06 12:12:23,657 - INFO - Building gene vocabulary


2026-08-06 12:12:25,133 - INFO - Instantiating Transcriptformer model
2026-08-06 12:12:25,133 - INFO - Instantiating the transcriptformer model


2026-08-06 12:12:27,072 - INFO - Model instantiated successfully
2026-08-06 12:12:27,072 - INFO - Loading model checkpoint


...

Predicting DataLoader 0: 100%|██████████| 27200/27200 [11:52<00:00, 38.19it/s]

2026-08-06 12:24:28,946 - INFO - Combining predictions


... storing 'orig.ident' as categorical
... storing 'cell_type' as categorical
... storing 'batch' as categorical
... storing 'celltype' as categorical
... storing 'NewCelltype' as categorical
... storing 'organism_ontology_term_id' as categorical
... storing 'assay_ontology_term_id' as categorical
... storing 'cell_type_ontology_term_id' as categorical
... storing 'assay' as categorical



Inference completed! Saved embeddings to data/results/cross_species_embedding/transcriptformer_metazoa_embeddings.h5ad


AnnData object with n_obs × n_vars = 27200 × 0
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'seurat_clusters', 'cell_type', 'batch', 'barcode', 'celltype', 'percent.mt', 'integrated_snn_res.1', 'NewCelltype', 'n_genes', 'organism_ontology_term_id', 'nnz', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'assay'
    obsm: 'embeddings', 'model_emb'

## scIB embedding scores

In [4]:
def build_benchmark_adata(
    expression_source: ad.AnnData,
    model_output: ad.AnnData,
    model_embedding_key: str,
) -> ad.AnnData:
    """Attach a model embedding to the full expression matrix and add common baselines."""
    if not expression_source.obs_names.equals(model_output.obs_names):
        raise RuntimeError("Expression source cell names/order differ from model output")
    if model_embedding_key not in model_output.obsm:
        raise KeyError(f"Missing model embedding: {model_embedding_key}")
    benchmark_adata = expression_source.copy()
    benchmark_adata.obsm[model_embedding_key] = np.asarray(
        model_output.obsm[model_embedding_key], dtype=np.float32
    )
    sc.pp.normalize_total(benchmark_adata, target_sum=1e4)
    sc.pp.log1p(benchmark_adata)
    sc.tl.pca(
        benchmark_adata,
        n_comps=50,
        svd_solver="arpack",
        use_highly_variable=False,
    )
    benchmark_adata.obsm["random"] = rng.random(
        benchmark_adata.obsm["X_pca"].shape, dtype=np.float32
    )
    return benchmark_adata


def benchmark_embeddings(adata: ad.AnnData, model_embedding_key: str):
    """Run the original cross-species scIB comparison and return unscaled scores."""
    for key in (BATCH_KEY, LABEL_KEY):
        if key not in adata.obs:
            raise KeyError(f"Missing obs column: {key}")
    benchmark = Benchmarker(
        adata,
        batch_key=BATCH_KEY,
        label_key=LABEL_KEY,
        embedding_obsm_keys=[model_embedding_key, "X_pca", "random"],
        pre_integrated_embedding_obsm_key="X_pca",
        bio_conservation_metrics=BioConservation(),
        batch_correction_metrics=BatchCorrection(),
        n_jobs=min(10, int(os.environ.get("SLURM_CPUS_PER_TASK", "10"))),
    )
    benchmark.benchmark()
    return benchmark.get_results(min_max_scale=False)

In [5]:
benchmark_da = build_benchmark_adata(
    sc.read_h5ad(DATA_PATH),
    output_da,
    "model_emb",
)
results = benchmark_embeddings(benchmark_da, "model_emb")
results.to_csv(SCORE_PATH)
output_da.obsm["X_pca"] = benchmark_da.obsm["X_pca"]
output_da.obsm["random"] = benchmark_da.obsm["random"]
output_da.write_h5ad(OUTPUT_PATH, compression="lzf")
display(results)
print("Scores:", SCORE_PATH)

/lustre/fswork/projects/rech/xeg/uat95fg/venvs/transcriptformer-0.6.1/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)


Computing neighbors:   0%|          | 0/3 [00:00<?, ?it/s]

Computing neighbors:  33%|███▎      | 1/3 [00:29<00:58, 29.30s/it]

Computing neighbors:  67%|██████▋   | 2/3 [00:33<00:14, 14.73s/it]

Computing neighbors: 100%|██████████| 3/3 [00:37<00:00,  9.50s/it]

...

Metrics:  90%|█████████ | 9/10 [00:17<00:01,  1.09s/it, Batch correction: pcr_comparison]

Embeddings: 100%|██████████| 3/3 [12:32<00:00, 174.07s/it]

Embeddings: 100%|██████████| 3/3 [12:32<00:00, 250.88s/it]

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,BRAS,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
model_emb,0.520022,0.502506,0.326535,0.522644,0.99908,0.669165,0.0,0.001951,0.89324,0.629601,0.438791,0.574157,0.520011
X_pca,0.600084,0.680804,0.578622,0.598215,1.0,0.496108,0.0,0.00017,0.897937,0.0,0.278843,0.691545,0.526464
random,0.497309,0.00097,-0.000109,0.497098,0.605265,0.993115,0.858914,0.927192,0.091494,0.999879,0.774119,0.320106,0.501711
Metric Type,Bio conservation,Bio conservation,Bio conservation,Bio conservation,Bio conservation,Batch correction,Batch correction,Batch correction,Batch correction,Batch correction,Aggregate score,Aggregate score,Aggregate score


Scores: data/results/cross_species_embedding/transcriptformer_metazoa_scib.csv
